# NYC Property Sales: Data Cleaning and Integration

## Objective

Clean and standardize the 2024-2025 NYC property sales files, preserve source provenance, apply defensible data-quality rules, and integrate the ten borough-level datasets into a single analytical dataset.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

files = sorted(RAW_DATA_DIR.glob("*/*.xlsx"))

print(f"Source files: {len(files)}")

Source files: 10


In [3]:
# load the datasets using the ingestion rule
datasets = {
    file.stem: pd.read_excel(file, header=6)
    for file in files
}

print(f"Datasets loaded: {len(datasets)}")

Datasets loaded: 10


## Column Standardization

The source files share a consistent schema, but several column names contain embedded line breaks, spaces, and punctuation. Standardized snake-case names are applied before further cleaning and integration.

In [4]:
# inspect current columns once
datasets["2024_bronx"].columns.tolist()

['BOROUGH',
 'NEIGHBORHOOD',
 'BUILDING CLASS CATEGORY',
 'TAX CLASS AT PRESENT',
 'BLOCK',
 'LOT',
 'EASE-MENT',
 'BUILDING CLASS AT PRESENT',
 'ADDRESS',
 'APARTMENT NUMBER',
 'ZIP CODE',
 'RESIDENTIAL\nUNITS',
 'COMMERCIAL\nUNITS',
 'TOTAL \nUNITS',
 'LAND \nSQUARE FEET',
 'GROSS \nSQUARE FEET',
 'YEAR BUILT',
 'TAX CLASS AT TIME OF SALE',
 'BUILDING CLASS\nAT TIME OF SALE',
 'SALE PRICE',
 'SALE DATE']

In [9]:
# standardize column names
def clean_column_name(column):
    return(
        column.strip()
        .lower()
        .replace("\n", " ")
        .replace("-", " ")
        .replace(" ", "_")
        .replace("__", "_")
        .replace("ease_ment", "easement")
    )

for name, df in datasets.items():
    df.columns = [clean_column_name(column) for column in df.columns]

In [10]:
# inspect the standardized result
datasets["2024_bronx"].columns.tolist()

['borough',
 'neighborhood',
 'building_class_category',
 'tax_class_at_present',
 'block',
 'lot',
 'easement',
 'building_class_at_present',
 'address',
 'apartment_number',
 'zip_code',
 'residential_units',
 'commercial_units',
 'total_units',
 'land_square_feet',
 'gross_square_feet',
 'year_built',
 'tax_class_at_time_of_sale',
 'building_class_at_time_of_sale',
 'sale_price',
 'sale_date']

In [11]:
# confim schema consistency remained intact
reference_columns = datasets["2024_bronx"].columns.tolist()

schema_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "same_columns": df.columns.tolist() == reference_columns,
            "column_count": len(df.columns),
        }
        for name, df in datasets.items()
    ]
)

schema_check

,dataset,same_columns,column_count
0,2024_bronx,True,21
1,2024_brooklyn,True,21
2,2024_manhattan,True,21
3,2024_queens,True,21
4,2024_staten_island,True,21
5,2025_bronx,True,21
6,2025_brooklyn,True,21
7,2025_manhattan,True,21
8,2025_queens,True,21
9,2025_staten_island,True,21


## Structural Cleaning

Each workbook contains one completely blank row immediately below the header. The `easement` field is also entirely missing across all source files. These structural issues can be removed without discarding observed transaction information.

In [12]:
# remove completely blank rows
blank_row_summary = []

for name, df in datasets.items():
    rows_before = len(df)
    datasets[name] = df.dropna(how="all").copy()
    rows_after = len(datasets[name])

    blank_row_summary.append(
        {
            "dataset": name,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "rows_removed": rows_before - rows_after,
        }
    )

blank_row_summary = pd.DataFrame(blank_row_summary)

blank_row_summary

,dataset,rows_before,rows_after,rows_removed
0,2024_bronx,6204,6203,1
1,2024_brooklyn,21995,21994,1
2,2024_manhattan,17380,17379,1
3,2024_queens,25004,25003,1
4,2024_staten_island,7665,7664,1
5,2025_bronx,6819,6818,1
6,2025_brooklyn,23473,23472,1
7,2025_manhattan,19725,19724,1
8,2025_queens,27259,27258,1
9,2025_staten_island,7797,7796,1


In [13]:
# verify `easement` before dropping
easement_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "non_missing_values": df["easement"].notna().sum(),
        }
        for name, df in datasets.items()
    ]
)

easement_check

,dataset,non_missing_values
0,2024_bronx,0
1,2024_brooklyn,0
2,2024_manhattan,0
3,2024_queens,0
4,2024_staten_island,0
5,2025_bronx,0
6,2025_brooklyn,0
7,2025_manhattan,0
8,2025_queens,0
9,2025_staten_island,0


In [15]:
# drop the entirely empty field
for name, df in datasets.items():
    datasets[name] = df.drop(columns="easement")

print(f"Columns after structural cleaning: {datasets['2024_bronx'].shape[1]}")

Columns after structural cleaning: 20


## Source Provenance

Year and borough labels are derived from the source fileneames and retained with each record. These fields preserve file-level provenance after integration and provide an independent check against the borough information contained in the source data.

In [16]:
# add source provenance
for name, df in datasets.items():
    year, borough = name.split("_", maxsplit=1)

    df["source_year"] = int(year)
    df["source_borough"] = borough.replace("_", " ").title()

In [17]:
# inspect provenance
provenance_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "source_year": df["source_year"].unique().tolist(),
            "source_borough": df["source_borough"].unique().tolist(),
            "borough_code": df["borough"].unique().tolist(),
        }
        for name, df in datasets.items()
    ]
)

provenance_summary

,dataset,source_year,source_borough,borough_code
0,2024_bronx,[2024],[Bronx],[2.0]
1,2024_brooklyn,[2024],[Brooklyn],[3.0]
2,2024_manhattan,[2024],[Manhattan],[1.0]
3,2024_queens,[2024],[Queens],[4.0]
4,2024_staten_island,[2024],[Staten Island],[5.0]
5,2025_bronx,[2025],[Bronx],[2.0]
6,2025_brooklyn,[2025],[Brooklyn],[3.0]
7,2025_manhattan,[2025],[Manhattan],[1.0]
8,2025_queens,[2025],[Queens],[4.0]
9,2025_staten_island,[2025],[Staten Island],[5.0]


In [19]:
# validate borough codes against source labels
borough_code_map = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island",
}

borough_validation = []

for name, df in datasets.items():
    expected_borough = df["source_borough"].iloc[0]
    observed_boroughs = (
        df["borough"]
        .map(borough_code_map)
        .dropna()
        .unique()
        .tolist()
    )

    borough_validation.append(
        {
            "dataset": name,
            "expected_borough": expected_borough,
            "observed_boroughs": observed_boroughs,
            "match": observed_boroughs == [expected_borough],
        }
    )

borough_validation = pd.DataFrame(borough_validation)

borough_validation

,dataset,expected_borough,observed_boroughs,match
0,2024_bronx,Bronx,[Bronx],True
1,2024_brooklyn,Brooklyn,[Brooklyn],True
2,2024_manhattan,Manhattan,[Manhattan],True
3,2024_queens,Queens,[Queens],True
4,2024_staten_island,Staten Island,[Staten Island],True
5,2025_bronx,Bronx,[Bronx],True
6,2025_brooklyn,Brooklyn,[Brooklyn],True
7,2025_manhattan,Manhattan,[Manhattan],True
8,2025_queens,Queens,[Queens],True
9,2025_staten_island,Staten Island,[Staten Island],True


In [20]:
# validate year against sale date
year_validation = pd.DataFrame(
    [
        {
            "dataset": name,
            "source_year": df["source_year"].iloc[0],
            "date_mismatches": (
                df["sale_date"].dt.year != df["source_year"]
            ).sum(),
        }
        for name, df in datasets.items()
    ]
)

year_validation

,dataset,source_year,date_mismatches
0,2024_bronx,2024,0
1,2024_brooklyn,2024,0
2,2024_manhattan,2024,0
3,2024_queens,2024,0
4,2024_staten_island,2024,0
5,2025_bronx,2025,0
6,2025_brooklyn,2025,0
7,2025_manhattan,2025,0
8,2025_queens,2025,0
9,2025_staten_island,2025,0


## Data Type Validation

Several identifier and count fields were initially inferred as floating-point values. Their observed values are checked before assigning more appropriate data types.

In [21]:
integer_candidates = [
    "borough",
    "block",
    "lot",
    "zip_code",
    "residential_units",
    "commercial_units",
    "total_units",
    "year_built",
    "tax_class_at_time_of_sale",
]

integer_validation = pd.DataFrame(
    [
        {
            "column": column,
            "non_missing_values": sum(
                df[column].notna().sum()
                for df in datasets.values()
            ),
            "fractional_values": sum(
                (
                    df[column].dropna()
                    % 1 != 0
                ).sum()
                for df in datasets.values()
            ),
        }
        for column in integer_candidates
    ]
)

integer_validation

,column,non_missing_values,fractional_values
0,borough,163311,0
1,block,163311,0
2,lot,163311,0
3,zip_code,163288,0
4,residential_units,123732,0
5,commercial_units,95864,0
6,total_units,129653,0
7,year_built,152532,0
8,tax_class_at_time_of_sale,163311,0


### Data Type Decisions

The candidate integer fields contain no fractional values. Count and year variables are therefore represented using nullable integer types where missing values remain possible.

Borough, block, lot, and ZIP code function primarily as identifiers rather than continuous measurements. They are represented accordingly to avoid implying arithmetic meaning.

In [22]:
# convert dtypes
for df in datasets.values():
    # Categorical/geographic identifiers
    df["borough"] = df["borough"].map(borough_code_map).astype("string")
    df["block"] = df["block"].astype("Int64").astype("string")
    df["lot"] = df["lot"].astype("Int64").astype("string")
    df["zip_code"] = df["zip_code"].astype("Int64").astype("string")

    # Nullable integer-valued measurements
    integer_columns = [
        "residential_units",
        "commercial_units",
        "total_units",
        "year_built",
        "tax_class_at_time_of_sale",
    ]

    df[integer_columns] = df[integer_columns].astype("Int64")

In [23]:
# verify
datasets["2024_bronx"].dtypes

borough                                   string
neighborhood                                 str
building_class_category                      str
tax_class_at_present                         str
block                                     string
lot                                       string
building_class_at_present                    str
address                                      str
apartment_number                             str
zip_code                                  string
residential_units                          Int64
commercial_units                           Int64
total_units                                Int64
land_square_feet                         float64
gross_square_feet                        float64
year_built                                 Int64
tax_class_at_time_of_sale                  Int64
building_class_at_time_of_sale               str
sale_price                               float64
sale_date                         datetime64[us]
source_year         

## Suspicious Value Investigation

Potentially implausible values are inspected in context before any records are removed or modified. This avoids treating unusual but valid observations as data errors solely because they are extreme.

In [26]:
# inspect unusually early construction years
early_year_records = pd.concat(
    [
        df.loc[
            df["year_built"].notna() & (df["year_built"] < 1700),
            [
                "source_year",
                "borough",
                "neighborhood",
                "building_class_category",
                "address",
                "year_built",
                "sale_price",
                "sale_date",
            ],
        ]
        for df in datasets.values()
    ],
    ignore_index=True,
)

early_year_records.sort_values("year_built")

,source_year,borough,neighborhood,building_class_category,address,year_built,sale_price,sale_date
0,2025,Manhattan,GREENWICH VILLAGE-WEST,13 CONDOS - ELEVATOR APARTMENTS,"118 WEST 13TH STREET, 04",190,10250000.0,2025-07-30


In [27]:
# frequency of suspicious years
early_year_records["year_built"].value_counts().sort_index()

year_built
190    1
Name: count, dtype: Int64

In [29]:
# aggregate sale-price diagnostics
sale_price_summary = pd.concat(
    [df["sale_price"] for df in datasets.values()],
    ignore_index=True,
).describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

sale_price_summary

count    1.633110e+05
mean     1.668226e+06
std      1.209879e+07
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
25%      0.000000e+00
50%      4.950000e+05
75%      9.999990e+05
95%      3.825000e+06
99%      1.939000e+07
max      1.080000e+09
Name: sale_price, dtype: float64

In [31]:
# count zero versus positive prices
sale_price_counts = pd.DataFrame(
    [
        {
            "dataset": name,
            "records": len(df),
            "zero_price": (df["sale_price"] == 0).sum(),
            "positive_price": (df["sale_price"] > 0).sum(),
        }
        for name, df in datasets.items()
    ]
)

sale_price_counts["zero_price_pct"] = (
    sale_price_counts["zero_price"]
        / sale_price_counts["records"]
        * 100
).round(2)

sale_price_counts

,dataset,records,zero_price,positive_price,zero_price_pct
0,2024_bronx,6203,2007,4196,32.36
1,2024_brooklyn,21994,8323,13671,37.84
2,2024_manhattan,17379,3965,13414,22.81
3,2024_queens,25003,9118,15885,36.47
4,2024_staten_island,7664,2558,5106,33.38
5,2025_bronx,6818,2395,4423,35.13
6,2025_brooklyn,23472,9171,14301,39.07
7,2025_manhattan,19724,4466,15258,22.64
8,2025_queens,27258,10709,16549,39.29
9,2025_staten_island,7796,2867,4929,36.78


In [33]:
# inspect the most expensive transactions
all_records_preview = pd.concat(
    datasets.values(),
    ignore_index=True,
)

all_records_preview.nlargest(
    20,
    "sale_price",
)[
    [
        "source_year",
        "borough",
        "neighborhood",
        "building_class_category",
        "address",
        "residential_units",
        "commercial_units",
        "land_square_feet",
        "gross_square_feet",
        "sale_price",
        "sale_date",
    ]
]

,source_year,borough,neighborhood,building_class_category,address,residential_units,commercial_units,land_square_feet,gross_square_feet,sale_price,sale_date
115785,2025,Manhattan,MIDTOWN CBD,21 OFFICE BUILDINGS,590 MADISON AVENUE,0,80,39162.0,999646.0,1.080000e+09,2025-08-14
115790,2025,Manhattan,MIDTOWN CBD,22 STORE BUILDINGS,9 EAST 56 STREET,0,1,5020.0,26983.0,1.080000e+09,2025-08-14
115791,2025,Manhattan,MIDTOWN CBD,22 STORE BUILDINGS,8 EAST 57 STREET,0,1,5020.0,26983.0,1.080000e+09,2025-08-14
35023,2024,Manhattan,MIDTOWN CBD,43 CONDO OFFICE BUILDINGS,"717 5 AVENUE, 4-OFF",<NA>,1,NaN,NaN,9.630000e+08,2024-01-22
35035,2024,Manhattan,MIDTOWN CBD,46 CONDO STORE BUILDINGS,"717 5 AVENUE, RETL",<NA>,1,NaN,NaN,9.630000e+08,2024-01-22
120537,2025,Manhattan,UPPER EAST SIDE (59-79),08 RENTALS - ELEVATOR APARTMENTS,800 5 AVENUE,208,26,28235.0,355978.0,8.100000e+08,2025-08-14
118068,2025,Manhattan,MIDTOWN WEST,21 OFFICE BUILDINGS,1177 AVENUE OF THE AMER,0,76,32134.0,912955.0,5.722903e+08,2025-09-15
40565,2024,Manhattan,UPPER EAST SIDE (59-79),21 OFFICE BUILDINGS,980 MADISON AVENUE,0,29,20390.0,118635.0,5.600000e+08,2024-06-05
122498,2025,Manhattan,UPPER EAST SIDE (59-79),43 CONDO OFFICE BUILDINGS,"1334 YORK AVENUE, DEM",<NA>,1,NaN,NaN,5.100000e+08,2025-10-01
122499,2025,Manhattan,UPPER EAST SIDE (59-79),43 CONDO OFFICE BUILDINGS,"1334 YORK AVENUE, 10",<NA>,1,NaN,NaN,5.100000e+08,2025-10-01


### Construction Year

One record contains a construction year of `190`, which is incomplete or invalid as a four-digit calendar year. Because the intended year cannot be determined reliably from the source data alone, the transaction is retained while the invalid construction-year value is set to missing.

In [34]:
# replace invalid construction year
invalid_year_count = 0

for df in datasets.values():
    invalid_mask = df["year_built"].notna() & (df["year_built"] < 1000)
    invalid_year_count += invalid_mask.sum()

    df.loc[invalid_mask, "year_built"] = pd.NA

print(f"Invalid construction years set to missing: {invalid_year_count}")

Invalid construction years set to missing: 1


In [36]:
min(
    df["year_built"].min()
    for df in datasets.values()
)

np.int64(1800)

## Residential Scope

The source data includes residential, commercial, mixed-use, and other property categories. Because the research question focuses on residential property sales, the analytical population must be defined explicitly from the observed building classifications before price analysis is performed.

In [38]:
# combine unique building categories
building_categories = sorted(
    pd.concat(
        [
            df["building_class_category"]
            for df in datasets.values()
        ],
        ignore_index=True,
    )
    .dropna()
    .unique()
)

print(f"Unique building class categories: {len(building_categories)}")

for category in building_categories:
    print(category)

Unique building class categories: 44
01 ONE FAMILY DWELLINGS
02 TWO FAMILY DWELLINGS
03 THREE FAMILY DWELLINGS
04 TAX CLASS 1 CONDOS
05 TAX CLASS 1 VACANT LAND
06 TAX CLASS 1 - OTHER
07 RENTALS - WALKUP APARTMENTS
08 RENTALS - ELEVATOR APARTMENTS
09 COOPS - WALKUP APARTMENTS
10 COOPS - ELEVATOR APARTMENTS
11 SPECIAL CONDO BILLING LOTS
12 CONDOS - WALKUP APARTMENTS
13 CONDOS - ELEVATOR APARTMENTS
14 RENTALS - 4-10 UNIT
15 CONDOS - 2-10 UNIT RESIDENTIAL
16 CONDOS - 2-10 UNIT WITH COMMERCIAL UNIT
17 CONDO COOPS
21 OFFICE BUILDINGS
22 STORE BUILDINGS
25 LUXURY HOTELS
26 OTHER HOTELS
27 FACTORIES
28 COMMERCIAL CONDOS
29 COMMERCIAL GARAGES
30 WAREHOUSES
31 COMMERCIAL VACANT LAND
32 HOSPITAL AND HEALTH FACILITIES
33 EDUCATIONAL FACILITIES
34 THEATRES
35 INDOOR PUBLIC AND CULTURAL FACILITIES
36 OUTDOOR RECREATIONAL FACILITIES
37 RELIGIOUS FACILITIES
38 ASYLUMS AND HOMES
39 TRANSPORTATION FACILITIES
40 SELECTED GOVERNMENTAL FACILITIES
41 TAX CLASS 4 - OTHER
42 CONDO CULTURAL/MEDICAL/EDUCATIONAL

### Residential Classification

The primary analysis is restricted to building categories that represent clearly residential property: one- to three-family dwellings, residential condominiums, apartment rentals, cooperatives, and residential condo/co-op categories.

Vacant land, explicitly mixed residential-commercial property, special billing lots, and commercial or institutional categories are excluded from the primary residential analytical population. This restriction defines the scope of the analysis rather than treating excluded records as data errors.

In [40]:
# define residential categories
residential_category_codes = {
    "01", "02", "03", "04",
    "07", "08", "09", "10",
    "12", "13", "14", "15", "17",
}

for df in datasets.values():
    df["building_category_code"] = (
        df["building_class_category"]
        .str[:2]
    )

    df["is_residential"] = (
        df["building_category_code"]
        .isin(residential_category_codes)
    )

In [41]:
# validate the classification
classification_check = (
    pd.concat(datasets.values(), ignore_index=True)
    .groupby(
        ["building_category_code", "building_class_category"],
        as_index=False,
    )
    .agg(
        records=("is_residential", "size"),
        is_residential=("is_residential", "first"),
    )
    .sort_values("building_category_code")
)

classification_check

,building_category_code,building_class_category,records,is_residential
0,01,01 ONE FAMILY DWELLINGS,35934,True
1,02,02 TWO FAMILY DWELLINGS,30111,True
2,03,03 THREE FAMILY DWELLINGS,8617,True
3,04,04 TAX CLASS 1 CONDOS,3111,True
4,05,05 TAX CLASS 1 VACANT LAND,1916,False
5,06,06 TAX CLASS 1 - OTHER,359,False
6,07,07 RENTALS - WALKUP APARTMENTS,5442,True
7,08,08 RENTALS - ELEVATOR APARTMENTS,943,True
8,09,09 COOPS - WALKUP APARTMENTS,4762,True
9,10,10 COOPS - ELEVATOR APARTMENTS,26205,True


In [42]:
# quantify the scope
residential_scope = pd.DataFrame(
    [
        {
            "dataset": name,
            "total_records": len(df),
            "residential_records": df["is_residential"].sum(),
        }
        for name, df in datasets.items()
    ]
)

residential_scope["residential_pct"] = (
    residential_scope["residential_records"]
    / residential_scope["total_records"]
    * 100
).round(2)

residential_scope

,dataset,total_records,residential_records,residential_pct
0,2024_bronx,6203,5574,89.86
1,2024_brooklyn,21994,19617,89.19
2,2024_manhattan,17379,16229,93.38
3,2024_queens,25003,23115,92.45
4,2024_staten_island,7664,6865,89.57
5,2025_bronx,6818,6110,89.62
6,2025_brooklyn,23472,21062,89.73
7,2025_manhattan,19724,18060,91.56
8,2025_queens,27258,25188,92.41
9,2025_staten_island,7796,7267,93.21


### Sale-Price Eligibility

Zero-price records are retained in the integrated dataset because they are part of the published source data. However, they are excluded from analyses that interpret `sale_price` as an observed transaction price.

A separate eligibility indicator is therefore used instead of deleting these records during general data cleaning.

In [43]:
for df in datasets.values():
    df["has_positive_sale_price"] = df["sale_price"] > 0

    df["is_residential_market_sale"] = (
        df["is_residential"]
        & df["has_positive_sale_price"]
    )

In [44]:
# quantify the resulting analytical population
market_sale_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "total_records": len(df),
            "residential_records": df["is_residential"].sum(),
            "residential_positive_price": df[
                "is_residential_market_sale"
            ].sum(),
        }
        for name, df in datasets.items()
    ]
)

market_sale_summary["retained_pct"] = (
    market_sale_summary["residential_positive_price"]
    / market_sale_summary["total_records"]
    * 100
).round(2)

market_sale_summary

,dataset,total_records,residential_records,residential_positive_price,retained_pct
0,2024_bronx,6203,5574,3793,61.15
1,2024_brooklyn,21994,19617,12033,54.71
2,2024_manhattan,17379,16229,12612,72.57
3,2024_queens,25003,23115,14555,58.21
4,2024_staten_island,7664,6865,4450,58.06
5,2025_bronx,6818,6110,3978,58.35
6,2025_brooklyn,23472,21062,12706,54.13
7,2025_manhattan,19724,18060,13963,70.79
8,2025_queens,27258,25188,15200,55.76
9,2025_staten_island,7796,7267,4559,58.48


In [45]:
# overall analytical scope
scope_totals = pd.Series(
    {
        "all_records": sum(len(df) for df in datasets.values()),
        "residential_records": sum(
            df["is_residential"].sum()
            for df in datasets.values()
        ),
        "residential_positive_price": sum(
            df["is_residential_market_sale"].sum()
            for df in datasets.values()
        ),
    }
)

scope_totals

all_records                   163311
residential_records           149087
residential_positive_price     97849
dtype: int64

### Analytical Population

Residential building categories account for most records in the source data. Requiring a positive recorded sale price further narrows the population used for price-based analysis.

Non-residential and zero-price records remain available in the integrated dataset. Analytical eligibility is represented through explicit flags so that filtering decisions remain transparent and reproducible.

In [47]:
# measure usable square-footage information
# within our residential positive-price population
size_coverage = pd.DataFrame(
    [
        {
            "dataset": name,
            "residential_positive_price": df[
                "is_residential_market_sale"
            ].sum(),
            "positive_gross_sqft": (
                df["is_residential_market_sale"]
                & df["gross_square_feet"].notna()
                & (df["gross_square_feet"] > 0)
            ).sum(),
            "positive_land_sqft": (
                df["is_residential_market_sale"]
                & df["land_square_feet"].notna()
                & (df["land_square_feet"] > 0)
            ).sum(),
        }
        for name, df in datasets.items()
    ]
)

size_coverage["gross_sqft_coverage_pct"] = (
    size_coverage["positive_gross_sqft"]
    / size_coverage["residential_positive_price"]
    * 100
).round(2)

size_coverage["land_sqft_coverage_pct"] = (
    size_coverage["positive_land_sqft"]
    / size_coverage["residential_positive_price"]
    * 100
).round(2)

size_coverage

,dataset,residential_positive_price,positive_gross_sqft,positive_land_sqft,gross_sqft_coverage_pct,land_sqft_coverage_pct
0,2024_bronx,3793,2338,2344,61.64,61.80
1,2024_brooklyn,12033,6613,6620,54.96,55.02
2,2024_manhattan,12612,864,867,6.85,6.87
3,2024_queens,14555,8323,8326,57.18,57.20
4,2024_staten_island,4450,3887,3896,87.35,87.55
5,2025_bronx,3978,2631,2641,66.14,66.39
6,2025_brooklyn,12706,6974,6985,54.89,54.97
7,2025_manhattan,13963,756,756,5.41,5.41
8,2025_queens,15200,8752,8755,57.58,57.60
9,2025_staten_island,4559,3973,3992,87.15,87.56


In [50]:
# gross-square-footage coverage by residential category
size_category_data = pd.concat(
    datasets.values(),
    ignore_index=True,
)

size_category_data = size_category_data[
    size_category_data["is_residential_market_sale"]
].copy()

size_by_category = (
    size_category_data
    .groupby(
        ["building_category_code", "building_class_category"],
        as_index=False,
    )
    .agg(
        records=("sale_price", "size"),
        gross_sqft_available=(
            "gross_square_feet",
            lambda x: (x > 0).sum(),
        ),
    )
)

size_by_category["gross_sqft_coverage_pct"] = (
    size_by_category["gross_sqft_available"]
    / size_by_category["records"]
    * 100
).round(2)

size_by_category

,building_category_code,building_class_category,records,gross_sqft_available,gross_sqft_coverage_pct
0,01,01 ONE FAMILY DWELLINGS,21186,20963,98.95
1,02,02 TWO FAMILY DWELLINGS,15579,15560,99.88
2,03,03 THREE FAMILY DWELLINGS,4102,4098,99.90
3,04,04 TAX CLASS 1 CONDOS,2024,0,0.00
4,07,07 RENTALS - WALKUP APARTMENTS,3165,3153,99.62
5,08,08 RENTALS - ELEVATOR APARTMENTS,645,644,99.84
6,09,09 COOPS - WALKUP APARTMENTS,4161,0,0.00
7,10,10 COOPS - ELEVATOR APARTMENTS,21408,0,0.00
8,12,12 CONDOS - WALKUP APARTMENTS,1312,0,0.00
9,13,13 CONDOS - ELEVATOR APARTMENTS,19310,0,0.00


In [49]:
# create size eligibility flag
for df in datasets.values():
    df["has_valid_gross_sqft"] = (
        df["gross_square_feet"].notna()
        & (df["gross_square_feet"] > 0)
    )

    df["is_size_analysis_eligible"] = (
        df["is_residential_market_sale"]
        & df["has_valid_gross_sqft"]
    )

### Property-Size Coverage

Gross square footage is nearly complete for one- to three-family dwellings and residential rental buildings, but it is unavailable for the condo and co-op categories in the source data. As a result, size-based analysis cannot represent the full residential sales population.

Property-size relationships and price per square foot will therefore be evaluated only for residential positive-price records with positive reported gross square footage. Findings from this subset must be interpreted as applying to properties with reported building area rather than to all NYC residential sales.

In [51]:
# calculate price per square foot
for df in datasets.values():
    df["price_per_sqft"] = pd.NA

    eligible = df["is_size_analysis_eligible"]

    df.loc[eligible, "price_per_sqft"] = (
        df.loc[eligible, "sale_price"]
        / df.loc[eligible, "gross_square_feet"]
    )

    df["price_per_sqft"] = pd.to_numeric(
        df["price_per_sqft"],
        errors="coerce",
    )

In [52]:
# validate the derived variable
price_per_sqft_validation = pd.DataFrame(
    [
        {
            "dataset": name,
            "eligible_records": df["is_size_analysis_eligible"].sum(),
            "calculated_values": df["price_per_sqft"].notna().sum(),
            "non_positive_values": (
                df["price_per_sqft"].dropna() <= 0
            ).sum(),
        }
        for name, df in datasets.items()
    ]
)

price_per_sqft_validation

,dataset,eligible_records,calculated_values,non_positive_values
0,2024_bronx,2338,2338,0
1,2024_brooklyn,6613,6613,0
2,2024_manhattan,864,864,0
3,2024_queens,8323,8323,0
4,2024_staten_island,3887,3887,0
5,2025_bronx,2631,2631,0
6,2025_brooklyn,6974,6974,0
7,2025_manhattan,756,756,0
8,2025_queens,8752,8752,0
9,2025_staten_island,3973,3973,0


## Data Integration

After structural cleaning, type standardization, provenance validation, and creation of analytical eligibility indicators, the ten borough-year datasets are combined row-wise into a single dataset.

In [53]:
property_sales = pd.concat(
    datasets.values(),
    ignore_index=True,
)

property_sales.shape

(163311, 29)

In [54]:
# inspect integrated structure
property_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 163311 entries, 0 to 163310
Data columns (total 29 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   borough                         163311 non-null  string        
 1   neighborhood                    163311 non-null  str           
 2   building_class_category         163311 non-null  str           
 3   tax_class_at_present            163311 non-null  str           
 4   block                           163311 non-null  string        
 5   lot                             163311 non-null  string        
 6   building_class_at_present       163311 non-null  str           
 7   address                         163311 non-null  str           
 8   apartment_number                39372 non-null   str           
 9   zip_code                        163288 non-null  string        
 10  residential_units               123732 non-null  Int64         
 11

In [55]:
# validate row preservation
expected_rows = sum(len(df) for df in datasets.values())

print(f"Expected rows: {expected_rows:,}")
print(f"Integrated rows: {len(property_sales):,}")
print(f"Rows preserved: {len(property_sales) == expected_rows}")

Expected rows: 163,311
Integrated rows: 163,311
Rows preserved: True


In [56]:
# final duplicate check
duplicate_count = property_sales.duplicated().sum()

print(f"Exact duplicate rows after integration: {duplicate_count:,}")

Exact duplicate rows after integration: 0


In [57]:
# final provenance counts
integrated_counts = (
    property_sales
    .groupby(
        ["source_year", "source_borough"],
        as_index=False,
    )
    .size()
    .sort_values(["source_year", "source_borough"])
)

integrated_counts

,source_year,source_borough,size
0,2024,Bronx,6203
1,2024,Brooklyn,21994
2,2024,Manhattan,17379
3,2024,Queens,25003
4,2024,Staten Island,7664
5,2025,Bronx,6818
6,2025,Brooklyn,23472
7,2025,Manhattan,19724
8,2025,Queens,27258
9,2025,Staten Island,7796


In [58]:
# Final analytical validation
final_validation = pd.Series(
    {
        "total_records": len(property_sales),
        "residential_records": property_sales[
            "is_residential"
        ].sum(),
        "residential_positive_price_records": property_sales[
            "is_residential_market_sale"
        ].sum(),
        "size_analysis_eligible_records": property_sales[
            "is_size_analysis_eligible"
        ].sum(),
        "price_per_sqft_values": property_sales[
            "price_per_sqft"
        ].notna().sum(),
    }
)

final_validation

total_records                         163311
residential_records                   149087
residential_positive_price_records     97849
size_analysis_eligible_records         45111
price_per_sqft_values                  45111
dtype: int64

## Processed Dataset

The validated integrated dataset is saved for use in the exploratory analysis notebook. Raw source files remain unchanged, while analytical scope and eligibility decisions are peserved as explicit variables in the processed data.

In [59]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "nyc_property_sales_2024_2025.csv"

property_sales.to_csv(
    output_path,
    index=False,
)

print(f"Processed dataset saved to: {output_path}")

Processed dataset saved to: ..\data\processed\nyc_property_sales_2024_2025.csv


In [61]:
# verify the saved file
saved_data = pd.read_csv(
    output_path,
    low_memory=False,
)

print(f"Saved rows: {len(saved_data):,}")
print(f"Saved columns: {saved_data.shape[1]}")
print(f"File exists: {output_path.exists()}")

Saved rows: 163,311
Saved columns: 29
File exists: True


# Cleaning and Integration Summary

The ten borough-level sales files were standardized and integrated into a single dataset containing 163,311 records and 29 variables. Structural blank rows and the entirely missing `easement` field were removed, identifier and count fields were assigned appropriate data types, and source year and borough information was preserved and validated. One invalid construction year was set to missing without removing the associated transaction.

The integrated dataset contains 149,087 records belonging to clearly residential building categories. Of these, 97,849 have positive recorded sale prices and form the primary population for sale-price analysis. Zero-price and non-residential records remain in the processed dataset and are identified through explicit analytical flags rather than being discarded.

Gross square footage is available for 45,111 residential positive-price records. Coverage is high for one- to three-family dwellings and residential rental buildings but unavailable for the retained condo and co-op categories. Consequently, property-size relationships and price-per-square-foot analysis will be restricted to the size-eligible subset and interpreted separately from the broader residential market.

Final validation confirmed that all expected records were preserved during integration, no exact duplicate rows were introduced, and every size-eligible record has a positive calculated price-per-square-foot value.